In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/small_bench/checkpoints/pre_cell_8.pickle

In [3]:
%%RecordEvent
%%time
### cell 8 ###

def combine_subset_of_data_from_multiple_years_20(question_of_interest, x_axis_title, include_2017=None):
    # Map each year to its responses DataFrame
    df_map = {
        '2017': responses_df_2017,
        '2018': responses_df_2018_cell10,
        '2019': responses_df_2019_cell10,
        '2020': responses_df_2020,
        '2021': responses_df_2021,
        '2022': responses_df_2022_cell10
    }
    # By default exclude 2017 unless flag is set
    years = list(df_map.keys())[1:] if include_2017 is None else list(df_map.keys())

    frames = []
    for year in years:
        s = df_map[year][question_of_interest]
        # Apply the 2018‐specific age‐bin replacement without mutating the original
        if year == '2018':
            s = s.replace(['70-79', '80+'], '70+')
        # Compute counts and percentages, then sort categories
        counts = s.value_counts(dropna=False)
        pct = (counts * 100 / s.count()).round(1).sort_index()
        # Build a small DataFrame for this year
        tmp = pct.to_frame('percentage')
        tmp['year'] = year
        frames.append(tmp)

    # Concatenate all years, add the x-axis column, and set final column names
    df_combined = pd.concat(frames)
    df_combined[x_axis_title] = df_combined.index
    df_combined.columns = ['percentage', 'year', x_axis_title]
    return df_combined

# Re‐define the question and x‐axis title (needed for the harness)
question_of_interest_cell20 = 'What is your age (# years)?'
title_for_x_axis_cell20 = ''

# Call the optimized combine function and display info
age_df_combined = combine_subset_of_data_from_multiple_years_20(
    question_of_interest_cell20,
    title_for_x_axis_cell20
)
age_df_combined.info()

<class 'pandas.core.frame.DataFrame'>
Index: 55 entries, 18-21 to 70+
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   percentage  55 non-null     float64
 1   year        55 non-null     object 
 2               55 non-null     object 
dtypes: float64(1), object(2)
memory usage: 1.7+ KB
CPU times: user 21 ms, sys: 154 µs, total: 21.1 ms
Wall time: 21.1 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_8_try_2.pickle